In [99]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

In [100]:
df = pd.read_csv("../data/raw/TTC Subway Delay Data since 2025.csv")
df.head()

,_id,Date,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Line,Vehicle
0,1,2025-01-01,02:10,Wednesday,BATHURST STATION,MUSAN,5,9,E,BD,5227
1,2,2025-01-01,02:30,Wednesday,DUNDAS STATION,MUIRS,0,0,NaN,YU,0
2,3,2025-01-01,02:32,Wednesday,BROADVIEW STATION,PUMST,0,0,E,BD,0
3,4,2025-01-01,02:58,Wednesday,KEELE STATION,EUSC,0,0,W,BD,5293
4,5,2025-01-01,02:58,Wednesday,COXWELL STATION,SUAE,0,0,NaN,BD,0


In [101]:
print(df.shape)
df.info()
df.isna().sum()

(45475, 11)
<class 'pandas.DataFrame'>
RangeIndex: 45475 entries, 0 to 45474
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   _id        45475 non-null  int64
 1   Date       45475 non-null  str  
 2   Time       45475 non-null  str  
 3   Day        45475 non-null  str  
 4   Station    45475 non-null  str  
 5   Code       45475 non-null  str  
 6   Min Delay  45475 non-null  int64
 7   Min Gap    45475 non-null  int64
 8   Bound      29615 non-null  str  
 9   Line       45298 non-null  str  
 10  Vehicle    45475 non-null  int64
dtypes: int64(4), str(7)
memory usage: 3.8 MB


_id              0
Date             0
Time             0
Day              0
Station          0
Code             0
Min Delay        0
Min Gap          0
Bound        15860
Line           177
Vehicle          0
dtype: int64

In [102]:
# Distribution of delay minutes
print(df["Min Delay"].describe())
print()
print("Share of 0-min delays:", (df["Min Delay"] == 0).mean().round(3))
print("Share of delays >= 5 min:", (df["Min Delay"] >= 5).mean().round(3))
print("Share of delays >= 10 min:", (df["Min Delay"] >= 10).mean().round(3))

count    45475.000000
mean         2.684354
std         12.669995
min          0.000000
25%          0.000000
50%          0.000000
75%          3.000000
max        900.000000
Name: Min Delay, dtype: float64

Share of 0-min delays: 0.652
Share of delays >= 5 min: 0.18
Share of delays >= 10 min: 0.058


In [103]:
# Incident counts by line
df["Line"].value_counts()

Line
YU                        23134
BD                        19758
SHP                        1772
YU/BD                       470
YUS/BD                       87
YUS                          13
YU/BD/SHP                    11
BD/YU                         7
BD/YUS                        5
SRT                           5
YU/ BD                        4
YUS/ BD                       3
YU / BD                       2
EC                            2
YUS / BD                      2
YU/BD LINES                   2
LINE 1                        1
95 YORK MILLS                 1
508 LAKESHORE                 1
116 MORNINGSIDE               1
YUS/ BD/ SHP                  1
62 MORTIMER                   1
LINE 2 - BLOOR DANFORT        1
YU -BD LINES                  1
29 DUFFERIN                   1
60 STEELES WEST               1
76 ROYAL YORK SOUTH           1
50 BURNHAMTHORPE              1
BD/YUS/SHP/FWLRT/ECLRT        1
506 CARLTON                   1
WOODBINE STN                  1
999

In [104]:
# Number of unique station names and the most frequent ones
print("Unique stations:", df["Station"].nunique())
df["Station"].value_counts().head(30)

Unique stations: 1251


Station
BLOOR STATION             1849
FINCH STATION             1582
KENNEDY BD STATION        1577
KIPLING STATION           1508
YONGE BD STATION          1109
EGLINTON STATION          1093
VMC STATION                944
UNION STATION              937
WILSON STATION             918
WARDEN STATION             835
VICTORIA PARK STATION      798
WELLESLEY STATION          785
BROADVIEW STATION          708
COXWELL STATION            662
ST CLAIR STATION           647
ST GEORGE YUS STATION      639
KEELE STATION              637
DAVISVILLE STATION         636
ST GEORGE BD STATION       631
DUNDAS WEST STATION        614
MAIN STREET STATION        612
COLLEGE STATION            610
SHEPPARD WEST STATION      605
ISLINGTON STATION          604
BATHURST STATION           559
ST CLAIR WEST STATION      537
SPADINA BD STATION         537
OSSINGTON STATION          537
QUEEN STATION              525
SHEPPARD-YONGE STATION     525
Name: count, dtype: int64

## Key findings so far
- 65% of incidents have a 0-minute delay; 18% are ≥ 5 min and 5.8% are ≥ 10 min.
- Target: classify delays as long if ≥ 5 min (better class balance than 10 min, and matches the common 5-minute threshold for reporting subway outages).
- `Line` is messy: multi-line combos written many ways, the old `YUS` code, and a few bus routes recorded by mistake. Needs cleaning.
- Max delay is 900 min, likely an outlier to inspect.

In [105]:
# How much of the data do the most frequent station names cover?
counts = df["Station"].value_counts()
for n in [50, 75, 100, 150]:
    coverage = counts.head(n).sum() / len(df)
    print(f"Top {n} names cover {coverage:.1%} of incidents")

Top 50 names cover 74.4% of incidents
Top 75 names cover 90.6% of incidents
Top 100 names cover 94.2% of incidents
Top 150 names cover 96.1% of incidents


In [106]:
# Look at the long tail: names outside the top 100
pd.set_option("display.max_rows", 100)
counts.iloc[100:160]

Station
FINCH STATION - PLATFO    32
KIPLING STATION PLATFO    32
YONGE UNIVERSITY SUBWA    31
VMC STATION (PLATFORM     30
BLOOR DANFORTH LINE 2     26
YONGE UNIVERSITY AND B    25
KEELE YARD                25
UNION STATION (TOWARDS    24
GREENWOOD CARHOUSE        24
LINE 2 - BLOOR DANFORT    23
VMC STATION PLATFORM 2    22
NORTH YORK CENTRE STAT    21
LINE 2 - BLOOR-DANFORT    20
LINE 1- YONGE UNIVERSI    20
BLOOR-DANFORTH LINE 2     19
SPADINA YU STATION        19
LINE 1 YONGE-UNIVERSIT    18
VMC STATION PLATFORM 1    18
GREENWOOD WYE             17
VMC STATION - PLATFORM    17
KIPLING STATION (APPRO    16
ST. GEORGE STATION        16
YONGE-UNIVERSITY LINE     16
KENNEDY BD STATION (AP    15
ISLINGTON STATION (APP    15
UNION TO KING STATIONS    15
LINE 1 - YONGE UNIVERS    14
ST GEORGE STATION         14
BLOOR DANFORTH LINE       14
LINE 2 BLOOR-DANFORTH     14
WILSON CARHOUSE           14
YONGE/UNIVERSITY AND B    13
DAVISVILLE CARHOUSE       13
YONGE STATION             13
UNION 

## Cleaning station names

The raw `Station` column has 1,251 unique values, but the subway has only about 75 stations.
Main problems: names truncated at 22 characters (e.g. "PIONEER VILLAGE STATIO"),
extra suffixes ("FINCH STATION - PLATFO"), line-wide incidents, segments between stations,
and typos.

Rules (applied in order), stored in a new column `station_clean`:
1. Normalize: uppercase, remove periods, treat "-" and "/" as spaces.
2. "X TO Y" → `BETWEEN`
3. Yonge-University or system-wide descriptions → `LINE-WIDE`
4. Any word starting with "STAT" → keep the text before it + " STATION"
5. Yards, carhouses, wyes → `YARD`
6. Other line descriptions → `LINE-WIDE`
7. Anything else, or any name with fewer than 20 incidents → `OTHER`

Result: 1,251 raw names → 86 cleaned categories.

In [107]:
import re

def clean_station(name):
    """Map a raw station name to a cleaned station or a catch-all category."""
    if pd.isna(name):
        return "UNKNOWN"

    # Normalize: uppercase, drop periods, treat "-" and "/" as spaces
    s = str(name).upper().replace(".", "")
    s = re.sub(r"[-/]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    # Segments between two stations, e.g. "UNION TO KING"
    if " TO " in s:
        return "BETWEEN"
    # Incidents across a whole line or the whole system
    if "YONGE UNIVERSITY" in s or "SYSTEM" in s or s == "YUS":
        return "LINE-WIDE"
    # Station names, including truncated ones like "PIONEER VILLAGE STATIO"
    match = re.search(r"\bSTAT", s)
    if match:
        return s[:match.start()].strip() + " STATION"
    # Yards, carhouses and track infrastructure
    if any(word in s for word in ["YARD", "CARHOUSE", "WYE"]):
        return "YARD"
    # Other line-level descriptions, e.g. "BLOOR DANFORTH LINE 2"
    if "LINE" in s or "SUBWA" in s:
        return "LINE-WIDE"
    return "OTHER"

df["station_clean"] = df["Station"].apply(clean_station)

# Rare leftovers (typos, one-off descriptions) go to OTHER
MIN_COUNT = 20
counts = df["station_clean"].value_counts()
rare = counts[counts < MIN_COUNT].index
df.loc[df["station_clean"].isin(rare), "station_clean"] = "OTHER"

print("Unique stations after cleaning:", df["station_clean"].nunique())

Unique stations after cleaning: 86


In [108]:
# Check the result: most common cleaned names, and how much falls into each catch-all group
print(df["station_clean"].value_counts().head(10))
print()
for group in ["YARD", "LINE-WIDE", "BETWEEN", "OTHER", "UNKNOWN"]:
    print(f"{group}: {(df['station_clean'] == group).mean():.1%}")

station_clean
BLOOR STATION         1862
FINCH STATION         1815
KENNEDY BD STATION    1780
KIPLING STATION       1729
LINE-WIDE             1195
YONGE BD STATION      1115
EGLINTON STATION      1103
VMC STATION           1097
UNION STATION         1043
WILSON STATION         931
Name: count, dtype: int64

YARD: 0.9%
LINE-WIDE: 2.6%
BETWEEN: 0.8%
OTHER: 1.4%
UNKNOWN: 0.0%


In [109]:
# What ended up in OTHER?
df.loc[df["station_clean"] == "OTHER", "Station"].value_counts().head(25)

Station
TORONTO TRANSIT COMMIS    78
SPADINA YU STATION        19
YONGE STATION             13
SCARBOROUGH CTR STATIO    12
TRACK LEVEL ACTIVITY       9
NORTH YORK CENTRE          8
DAVISVILLE BUILD UP        8
EARLY ACCESS CLOSURE -     8
OSSINGTON CENTRE TRACK     7
DAVISVILLE BUILD-UP        6
MAIN STATION               5
MCBRIEN BUILDING           5
GUNN BUILDING              5
LEASIDE STATION            5
VARIOUS LOCATIONS          5
TEDDINGTON EE              4
SOUTH HOSTLER              4
DANFORTH DIVISION          4
YONGE AND BLOOR            4
GREENWOOD PORTAL           4
WILSON HOSTLER             4
MARKDALE EE                4
EGLINTON BUS TERMINAL      4
BLOOR DANFORTH             4
GREENWOOD CAR HOUSE        3
Name: count, dtype: int64

In [110]:
# Cleaned station names that appear rarely: likely leftover variants
clean_counts = df["station_clean"].value_counts()
clean_counts[clean_counts < 50]

station_clean
ST GEORGE STATION            35
NORTH YORK CENTRE STATION    21
Name: count, dtype: int64

## Cleaning line names

In [111]:
def clean_line(line):
    """Map a raw line value to YU, BD, SHP, MULTI, OTHER or UNKNOWN."""
    if pd.isna(line):
        return "UNKNOWN"
    s = re.sub(r"\s+", "", str(line).upper())  # remove all spaces

    if s in ("YU", "YUS") or s.startswith("LINE1"):
        return "YU"
    if s == "BD" or s.startswith("LINE2"):
        return "BD"
    if s == "SHP" or s.startswith("LINE4"):
        return "SHP"
    # Combinations of lines, e.g. "YU/BD", "YUS / BD", "YU/BD LINES"
    if "/" in s or "-" in s or "LINES" in s:
        return "MULTI"
    # Bus routes recorded by mistake, the closed SRT, and other one-offs
    return "OTHER"

df["line_clean"] = df["Line"].apply(clean_line)
df["line_clean"].value_counts()

line_clean
YU         23148
BD         19760
SHP         1772
MULTI        597
UNKNOWN      177
OTHER         21
Name: count, dtype: int64

In [112]:
# Which raw values ended up in OTHER?
df.loc[df["line_clean"] == "OTHER", "Line"].value_counts()

Line
SRT                    5
EC                     2
95 YORK MILLS          1
508 LAKESHORE          1
116 MORNINGSIDE        1
62 MORTIMER            1
29 DUFFERIN            1
60 STEELES WEST        1
76 ROYAL YORK SOUTH    1
50 BURNHAMTHORPE       1
506 CARLTON            1
WOODBINE STN           1
999                    1
504 KING               1
47 LANSDOWNE           1
TRANSIT CONTROL        1
Name: count, dtype: int64

In [113]:
# Fill missing lines using the most common line for each station
known = df[df["line_clean"].isin(["YU", "BD", "SHP"])]
station_to_line = known.groupby("station_clean")["line_clean"].agg(lambda x: x.mode()[0])

missing = df["line_clean"] == "UNKNOWN"
df.loc[missing, "line_clean"] = (
    df.loc[missing, "station_clean"].map(station_to_line).fillna("UNKNOWN")
)
df["line_clean"].value_counts()

line_clean
YU       23268
BD       19815
SHP       1774
MULTI      597
OTHER       21
Name: count, dtype: int64

Line cleaning results:
- `YU` (incl. `YUS`, `LINE 1`), `BD` (incl. `LINE 2`), `SHP`: 98.6% of incidents.
- `MULTI`: 597 incidents affecting several lines; kept as its own category.
- `OTHER`: 21 rows (bus routes, the closed SRT, misc codes); to be dropped.
- `UNKNOWN`: 177 missing values, all filled with the most common line of the same station.

## Delay cause codes

In [114]:
# Load the code descriptions
codes = pd.read_csv("../data/raw/Code Descriptions.csv")

def fix_encoding(text):
    """Repair text that was decoded with the wrong encoding, e.g. 'â\x80\x93' -> '–'."""
    try:
        return text.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError, AttributeError):
        return text  # leave it unchanged if it cannot be repaired

codes["DESCRIPTION"] = codes["DESCRIPTION"].apply(fix_encoding)

print(codes.shape)
codes.head(10)

(140, 3)


,_id,CODE,DESCRIPTION
0,1,EUAC,AIR CONDITIONING
1,2,EUAL,ALTERNATING CURRENT
2,3,EUATC,ATC RC&S EQUIPMENT
3,4,EUBK,BRAKES
4,5,EUBO,BODY
5,6,EUCA,COMPRESSED AIR
6,7,EUCC,CAM CONTROL
7,8,EUCD,RC&S CONSEQUENTIAL DELAY (SECOND DELAY SAME FA...
8,9,EUCH,CHOPPER CONTROL
9,10,EUCO,COUPLERS


In [115]:
# Do all codes in the delay data have a description?
print("Codes used in delay data:", df["Code"].nunique())
print("Codes in description table:", codes["CODE"].nunique())

missing_codes = set(df["Code"]) - set(codes["CODE"])
print("Codes without a description:", len(missing_codes))
df.loc[df["Code"].isin(missing_codes), "Code"].value_counts().head(15)

Codes used in delay data: 140
Codes in description table: 140
Codes without a description: 10


Code
SUL      1132
MUINF     202
PUTO      199
MUSE       38
PUFA       32
MUFL       27
MUBIO      13
XXXXX       9
TFPD        1
STAE        1
Name: count, dtype: int64

In [116]:
# Check the prefix idea: first two letters of each code
df["code_prefix"] = df["Code"].str[:2]
df["code_prefix"].value_counts()

code_prefix
MU    18641
SU    12759
PU     7160
TU     4382
EU     2522
XX        9
TF        1
ST        1
Name: count, dtype: int64

In [117]:
# What kinds of causes does each prefix cover? Show a few descriptions per prefix
codes["prefix"] = codes["CODE"].str[:2]
for prefix, group in codes.groupby("prefix"):
    print(f"{prefix} ({len(group)} codes):")
    print("   ", "; ".join(group["DESCRIPTION"].head(8)))
    print()

EU (30 codes):
    AIR CONDITIONING; ALTERNATING CURRENT; ATC RC&S EQUIPMENT; BRAKES; BODY; COMPRESSED AIR; CAM CONTROL; RC&S CONSEQUENTIAL DELAY (SECOND DELAY SAME FAULT – NO INTERVENTION)

MU (34 codes):
    ATC PROJECT; DIVISIONAL CLERK RELATED; CLOSURES - PLANNED; NO COLLECTOR AVAILABLE- E.S.A. RELATED; CLOSURES - UNPLANNED; DOOR PROBLEMS – PASSENGER RELATED; DOOR PROBLEMS – DEBRIS RELATED; ENGINEEERING & CONSTRUCTION RELATED

PU (45 codes):
    ATC SIGNALS OTHER; CENTRAL LOGIC CONTROLLER FAILURE; SIGNAL CONTROL; CENTRAL OFFICE SIGNALLING SYSTEM (CSS); DATA COMMUNICATIONS SYSTEM FAILURE; ELECTRICAL – MAINTENANCE ERROR; ELECTRICAL – OTHER; ELECTRICAL – WORK ZONE PROBLEMS

SU (13 codes):
    ASSAULT / EMPLOYEE INVOLVED; ASSUALT / PATRON INVOLVED; BOMB THREAT; COLLECTOR BOOTH HOLD-UP ALARM ACTIVATED; DISORDERLY PATRON; EAS ACTIVATED – NO TROUBLE FOUND; GRAFFITI / SCRATCHITI; SECURITY OTHER

TU (18 codes):
    ATC OPERATOR RELATED; TRANSIT CONTROL RELATED – TOWER CONTROLLER ERROR; DOOR

In [118]:
# Target: a delay is "long" if it lasts 5 minutes or more
df["long_delay"] = (df["Min Delay"] >= 5).astype(int)

# Group rare prefixes into OTHER
main_prefixes = ["MU", "SU", "PU", "TU", "EU"]
df["cause_group"] = df["code_prefix"].where(df["code_prefix"].isin(main_prefixes), "OTHER")

# Share of long delays and number of incidents for each cause group
df.groupby("cause_group")["long_delay"].agg(["mean", "count"]).sort_values("mean", ascending=False)

,mean,count
cause_group,,
EU,0.295004,2522
PU,0.211872,7160
SU,0.186613,12759
TU,0.184162,4382
OTHER,0.181818,11
MU,0.145486,18641


In [119]:
# Long-delay share for individual codes (only codes with enough incidents)
code_stats = (
    df.groupby("Code")["long_delay"]
    .agg(["mean", "count"])
    .query("count >= 100")
    .merge(codes[["CODE", "DESCRIPTION"]], left_index=True, right_on="CODE", how="left")
    .sort_values("mean", ascending=False)
)
code_stats[["CODE", "DESCRIPTION", "count", "mean"]]

,CODE,DESCRIPTION,count,mean
88.0,PUSSW,TRACK SWITCH FAILURE – S/E/C RELATED PROBLEM,101,0.693069
102.0,PUTR,RAIL RELATED PROBLEM,226,0.685841
42.0,MUI,INJURED/ILL CUSTOMER ON TRAIN – TRANSPORTED,668,0.618263
108.0,PUTWZ,WORK ZONES PROBLEMS – TRACK,369,0.617886
18.0,EUO,RC&S OTHER,120,0.575000
28.0,EUVE,WORK VEHICLE,110,0.572727
NaN,PUTO,NaN,199,0.562814
123.0,TUCC,TRANSIT CONTROL RELATED – TOWER CONTROLLER ERROR,140,0.557143
17.0,EUNT,EQUIPMENT – NO TROUBLE FOUND,253,0.529644
36.0,MUDD,DOOR PROBLEMS – DEBRIS RELATED,377,0.485411


Cause groups (from the first two letters of the code, checked against the code descriptions):
- `EU` vehicle equipment, `PU` infrastructure (signals, power), `SU` security,
  `TU` operations staff, `MU` miscellaneous (closures, passenger-related door problems, construction).
- Long-delay share by group: EU 29.5%, PU 21.2%, SU 18.7%, TU 18.4%, MU 14.5% (overall 18%).
- 10 codes used in the data have no description (3.6% of incidents, mostly `SUL`), but still have a valid prefix.

In [120]:
# Feature: keep frequent codes, group rare ones into "<prefix>_OTHER"
MIN_CODE_COUNT = 100
code_counts = df["Code"].value_counts()
frequent_codes = code_counts[code_counts >= MIN_CODE_COUNT].index

df["code_feature"] = df["Code"].where(
    df["Code"].isin(frequent_codes),
    df["cause_group"] + "_OTHER",
)
print("Number of code categories:", df["code_feature"].nunique())
print("Share of incidents in a *_OTHER group:",
      f"{df['code_feature'].str.endswith('_OTHER').mean():.1%}")

Number of code categories: 60
Share of incidents in a *_OTHER group: 5.7%


Individual codes are much more informative than cause groups:
- Long-delay share ranges from ~69% (track switch failure, rail problems) to 0%
  (planned closures, no collector available, escalator/stairway incidents).
- Location matters within the same cause: an injured customer transported from a train
  leads to a long delay 62% of the time, but only 0.7% when it happens in a station.
- Several codes never cause a long delay because they are not train delays at all.
  They are kept for now, but evaluating the model on train-affecting incidents only
  is a possible extension.
- Feature `code_feature`: codes with ≥ 100 incidents are kept; rarer codes are grouped
  into `<prefix>_OTHER`.
- Code descriptions had encoding errors (e.g. "â" instead of "–"); repaired with `fix_encoding`.